In [1]:
import os
import pandas as pd
import hashlib
from pathlib import Path


PROJECT_DIR = Path("..")
DATA_DIR = PROJECT_DIR / "data"
EVIDENCE_DIR = PROJECT_DIR / "evidence"

print("Project directory:", PROJECT_DIR.resolve())
print("Data directory:", DATA_DIR.resolve())
print("Evidence directory:", EVIDENCE_DIR.resolve())

Project directory: C:\Users\rajuh\Downloads\Challenge_2_Log_Analysis
Data directory: C:\Users\rajuh\Downloads\Challenge_2_Log_Analysis\data
Evidence directory: C:\Users\rajuh\Downloads\Challenge_2_Log_Analysis\evidence


In [2]:
files = sorted(DATA_DIR.iterdir())

for f in files:
    print(f"{f.name:40} {f.stat().st_size / (1024**2):8.2f} MB")

endpoint_security.csv                        3.11 MB
SHA256SUMS                                   0.00 MB
usb_events.csv                               0.00 MB
web_activity_part1.csv                      60.47 MB
web_activity_part2.csv                      60.42 MB
web_activity_part3.csv                      60.41 MB
web_activity_part4.csv                      60.46 MB
web_activity_part5.csv                      60.45 MB


In [3]:

security_file = DATA_DIR / "endpoint_security.csv"

security_df = pd.read_csv(security_file)

print("Rows:", len(security_df))
print("Columns:", list(security_df.columns))
print("\nLog types:")
print(security_df["log_type"].value_counts())

print("\nFirst 5 records:")
display(security_df.head())

Rows: 23540
Columns: ['endpoint_id', 'endpoint_name', 'mac', 'unit', 'log_type', 'event_time', 'received_time', 'message']

Log types:
log_type
quarantine    20004
av_scan        2622
web_block       914
Name: count, dtype: int64

First 5 records:


,endpoint_id,endpoint_name,mac,unit,log_type,event_time,received_time,message
0,EPZRAPEL,host-d14b,8A:58:3E:18:FD:1B,unit-foxtrot,quarantine,01-09-2026 04:58:58,01-09-2026 04:58:56,No suspicious file found by type or extension.
1,EPZRAPEL,host-d14b,8A:58:3E:18:FD:1B,unit-foxtrot,quarantine,01-09-2026 04:58:58,01-09-2026 04:58:56,File: 53EDA42869D426262B752F088EA605187284ACE8...
2,EPYJFFRM,host-2864,F2:EA:9C:78:DC:3C,unit-oscar,quarantine,01-09-2026 05:20:37,01-09-2026 05:20:39,No suspicious file found by type or extension.
3,EPYJFFRM,host-2864,F2:EA:9C:78:DC:3C,unit-oscar,quarantine,01-09-2026 05:20:37,01-09-2026 05:20:39,No Malicious files found.
4,EPFFTO5T,host-508b,FA:2F:46:B7:14:23,unit-delta,quarantine,01-09-2026 08:50:38,01-09-2026 08:50:38,No suspicious file found by type or extension.


In [4]:

usb_file = DATA_DIR / "usb_events.csv"

usb_df = pd.read_csv(usb_file)

print("USB rows:", len(usb_df))
print("\nColumns:")
print(list(usb_df.columns))

display(usb_df)

USB rows: 8

Columns:
['endpoint_id', 'endpoint_name', 'mac', 'unit', 'log_type', 'event_time', 'received_time', 'message']


,endpoint_id,endpoint_name,mac,unit,log_type,event_time,received_time,message
0,EPD24DBC,host-7394,6E:DE:00:D5:9F:8A,unit-foxtrot,usb,22-09-2026 18:23:12,22-09-2026 19:03:03,"General -UDisk , Mass Storage abcd:1234"
1,EPD24DBC,host-7394,6E:DE:00:D5:9F:8A,unit-foxtrot,usb,22-09-2026 18:23:41,22-09-2026 19:03:03,"General -UDisk , Mass Storage abcd:1234"
2,EPD24DBC,host-7394,6E:DE:00:D5:9F:8A,unit-foxtrot,usb,22-09-2026 18:23:12,23-09-2026 09:05:16,"General -UDisk , Mass Storage abcd:1234"
3,EPD24DBC,host-7394,6E:DE:00:D5:9F:8A,unit-foxtrot,usb,22-09-2026 18:23:41,23-09-2026 09:05:16,"General -UDisk , Mass Storage abcd:1234"
4,EP7FZ2XY,host-9764,F6:6F:85:06:00:26,unit-quebec,usb,07-09-2026 22:34:11,07-09-2026 09:03:19,"usb-usage: Logitech-USB Optical Mouse, Human I..."
5,EP7FZ2XY,host-9764,F6:6F:85:06:00:26,unit-quebec,usb,07-09-2026 22:34:12,07-09-2026 09:03:19,"usb-usage: Logitech-USB Optical Mouse, Human I..."
6,EP7FZ2XY,host-9764,F6:6F:85:06:00:26,unit-quebec,usb,07-09-2026 22:34:13,07-09-2026 09:03:19,"usb-usage: Logitech-USB Optical Mouse, Human I..."
7,EP7FZ2XY,host-9764,F6:6F:85:06:00:26,unit-quebec,usb,07-09-2026 22:34:14,07-09-2026 09:03:19,"usb-usage: Logitech-USB Optical Mouse, Human I..."


In [5]:


malicious_quarantine = security_df[
    (security_df["log_type"] == "quarantine") &
    (security_df["message"].str.contains(
        "FileType: Malicious",
        case=False,
        na=False
    ))
].copy()

print("Malicious quarantine records:", len(malicious_quarantine))
print("Unique endpoints:", malicious_quarantine["endpoint_id"].nunique())

display(
    malicious_quarantine[
        [
            "endpoint_id",
            "endpoint_name",
            "unit",
            "event_time",
            "message"
        ]
    ].head(20)
)

Malicious quarantine records: 134
Unique endpoints: 33


,endpoint_id,endpoint_name,unit,event_time,message
1,EPZRAPEL,host-d14b,unit-foxtrot,01-09-2026 04:58:58,File: 53EDA42869D426262B752F088EA605187284ACE8...
394,EPOC6Y4R,host-e783,unit-charlie,01-09-2026 08:59:34,File: recently-used.xbel | Quarantined At: 202...
671,EPOC6Y4R,host-e783,unit-charlie,01-09-2026 18:21:26,File: recently-used.xbel | Quarantined At: 202...
995,EPJYEC7F,host-38aa,unit-romeo,01-09-2026 08:09:22,File: recently-used.xbel | Quarantined At: 202...
1045,EPMQTKSL,host-942d,unit-charlie,02-09-2026 08:37:11,File: recently-used.xbel | Quarantined At: 202...
1050,EPOC6Y4R,host-e783,unit-charlie,02-09-2026 08:39:53,File: recently-used.xbel | Quarantined At: 202...
1148,EPC2L3UH,host-b4da,unit-india,02-09-2026 08:48:22,File: recently-used.xbel | Quarantined At: 202...
2843,EPOC6Y4R,host-e783,unit-charlie,04-09-2026 08:23:22,File: recently-used.xbel | Quarantined At: 202...
2997,EPAGTKNF,host-227c,unit-quebec,04-09-2026 18:10:01,"File: Buy CP PLUS Dashboard Camera, Warranty N..."
3415,EPAGTKNF,host-227c,unit-quebec,03-09-2026 19:03:33,"File: Buy CP PLUS Dashboard Camera, Warranty N..."


In [6]:

security_df["event_time_dt"] = pd.to_datetime(
    security_df["event_time"],
    format="%d-%m-%Y %H:%M:%S",
    errors="coerce"
)

malicious_quarantine["event_time_dt"] = pd.to_datetime(
    malicious_quarantine["event_time"],
    format="%d-%m-%Y %H:%M:%S",
    errors="coerce"
)

# Find the first malicious quarantine event for every endpoint
compromised_summary = (
    malicious_quarantine
    .sort_values("event_time_dt")
    .groupby("endpoint_id", as_index=False)
    .first()
)

# Keep the useful columns
compromised_summary = compromised_summary[
    [
        "endpoint_id",
        "endpoint_name",
        "unit",
        "event_time",
        "message"
    ]
].sort_values("event_time")

print("Confirmed endpoints with malicious quarantine evidence:",
      len(compromised_summary))

display(compromised_summary)

Confirmed endpoints with malicious quarantine evidence: 33


,endpoint_id,endpoint_name,unit,event_time,message
32,EPZRAPEL,host-d14b,unit-foxtrot,01-09-2026 04:58:58,File: 53EDA42869D426262B752F088EA605187284ACE8...
14,EPJYEC7F,host-38aa,unit-romeo,01-09-2026 08:09:22,File: recently-used.xbel | Quarantined At: 202...
20,EPOC6Y4R,host-e783,unit-charlie,01-09-2026 08:59:34,File: recently-used.xbel | Quarantined At: 202...
17,EPMQTKSL,host-942d,unit-charlie,02-09-2026 08:37:11,File: recently-used.xbel | Quarantined At: 202...
6,EPC2L3UH,host-b4da,unit-india,02-09-2026 08:48:22,File: recently-used.xbel | Quarantined At: 202...
4,EPAGTKNF,host-227c,unit-quebec,03-09-2026 19:03:33,"File: Buy CP PLUS Dashboard Camera, Warranty N..."
12,EPJ42VCZ,host-56a6,unit-india,07-09-2026 08:43:44,File: RAGHAV STNC STNA.pdf | Quarantined At: 2...
21,EPPGWIOF,host-a656,unit-india,07-09-2026 11:25:38,File: rawtobr3 | Quarantined At: 2026-09-07 10...
23,EPQKOBYG,host-d29a,unit-romeo,08-09-2026 08:31:09,File: win-lbp623-621-fw-v1301(1).exe | Quarant...
16,EPMDU6L5,host-b05e,unit-oscar,08-09-2026 11:58:09,File: recently-used.xbel | Quarantined At: 202...


In [8]:

window_start = pd.Timestamp("2026-09-10 00:00:00")
window_end = pd.Timestamp("2026-09-21 23:59:59")


first_malicious = (
    malicious_quarantine
    .sort_values("event_time_dt")
    .groupby("endpoint_id", as_index=False)
    .first()
)


window_compromised = first_malicious[
    (first_malicious["event_time_dt"] >= window_start) &
    (first_malicious["event_time_dt"] <= window_end)
].copy()

window_compromised = window_compromised.sort_values("event_time_dt")

print("First detected during web-activity window:",
      len(window_compromised))

display(
    window_compromised[
        [
            "endpoint_id",
            "endpoint_name",
            "unit",
            "event_time",
            "message"
        ]
    ].to_string(index=False)
)

First detected during web-activity window: 14


'endpoint_id endpoint_name         unit          event_time                                                                                                                                                                                       message\n   EPR4BEDS     host-fe94   unit-romeo 10-09-2026 09:01:45                                  File: recently-used.xbel | Quarantined At: 2026-09-09 13:15 | FileType: Malicious | SHA256: 5e10b3490160cca0fc872f2b026d4b40bafc236852f83d30c9031899e50adc9c\n   EPEEHWSG     host-4724    unit-kilo 10-09-2026 09:15:06                                  File: Library Books.xlsx | Quarantined At: 2026-09-08 21:19 | FileType: Malicious | SHA256: 98d119993c27068f00bdf18a17d6613ae6ce6046b6b6620979282c2662c4efbd\n   EP4YIFHV     host-a616    unit-lima 11-09-2026 09:23:33                                      File: webui-setup.js | Quarantined At: 2026-09-10 13:30 | FileType: Malicious | SHA256: e3d95016d065059105580ae8ba811cf4bb130b515b8c77e5e58cf95763ad938a\

In [10]:
# Clean table of first malicious quarantine event per endpoint

clean_quarantine = first_quarantine[
    [
        "endpoint_id",
        "endpoint_name",
        "unit",
        "quarantined_at",
        "message"
    ]
].copy()

# Extract filename and SHA256 separately
clean_quarantine["file_name"] = clean_quarantine["message"].str.extract(
    r"File:\s*(.*?)\s*\|\s*Quarantined At:",
    expand=False
)

clean_quarantine["sha256"] = clean_quarantine["message"].str.extract(
    r"SHA256:\s*([a-fA-F0-9]{64})",
    expand=False
)

# Select only useful evidence columns
clean_quarantine = clean_quarantine[
    [
        "endpoint_id",
        "endpoint_name",
        "unit",
        "quarantined_at",
        "file_name",
        "sha256"
    ]
].sort_values("quarantined_at")

print("First malicious quarantine event for each endpoint:")
print("Total endpoints:", len(clean_quarantine))

display(clean_quarantine)

First malicious quarantine event for each endpoint:
Total endpoints: 33


,endpoint_id,endpoint_name,unit,quarantined_at,file_name,sha256
16,EPMDU6L5,host-b05e,unit-oscar,2026-05-13 13:24:00,recently-used.xbel,5dd59e8309b4be90aad58871d82a5b3fb6dcd7ff19a7fb...
14,EPJYEC7F,host-38aa,unit-romeo,2026-08-31 13:17:00,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...
32,EPZRAPEL,host-d14b,unit-foxtrot,2026-08-31 13:21:00,53EDA42869D426262B752F088EA605187284ACE8,2e4dff1672bbabba74a0a99d54e38e56e036ce84719895...
20,EPOC6Y4R,host-e783,unit-charlie,2026-08-31 13:35:00,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...
6,EPC2L3UH,host-b4da,unit-india,2026-09-01 13:17:00,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...
17,EPMQTKSL,host-942d,unit-charlie,2026-09-01 13:56:00,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...
4,EPAGTKNF,host-227c,unit-quebec,2026-09-03 11:25:00,"Buy CP PLUS Dashboard Camera, Warranty NONE 1 ...",18866b6f7308f73bc1b63193e50629bfd4fff701d6b9d3...
12,EPJ42VCZ,host-56a6,unit-india,2026-09-06 10:20:00,RAGHAV STNC STNA.pdf,a2beca1870ae689f4b046a20c3d8e2e48557d782dea534...
21,EPPGWIOF,host-a656,unit-india,2026-09-07 10:57:00,brprintconflsr3,6594cac08d03782ad43d6b293faff904f20032c8eb37a1...
23,EPQKOBYG,host-d29a,unit-romeo,2026-09-07 13:57:00,win-lbp623-621-fw-v1301(1).exe,f475a7462dd85932ad7f99354751663d1543adb5e37c7c...


In [11]:
# Clean list of endpoints first quarantined during the web-activity window

window_quarantines_clean = first_quarantine[
    (first_quarantine["quarantined_at"] >= window_start) &
    (first_quarantine["quarantined_at"] <= window_end)
].copy()

window_quarantines_clean["file_name"] = (
    window_quarantines_clean["message"]
    .str.extract(
        r"File:\s*(.*?)\s*\|\s*Quarantined At:",
        expand=False
    )
)

window_quarantines_clean["sha256"] = (
    window_quarantines_clean["message"]
    .str.extract(
        r"SHA256:\s*([a-fA-F0-9]{64})",
        expand=False
    )
)

window_quarantines_clean = window_quarantines_clean[
    [
        "endpoint_id",
        "endpoint_name",
        "unit",
        "quarantined_at",
        "file_name",
        "sha256"
    ]
].sort_values("quarantined_at")

print("Endpoints first quarantined during 10–21 September:")
print("Total:", len(window_quarantines_clean))

display(window_quarantines_clean)

Endpoints first quarantined during 10–21 September:
Total: 13


,endpoint_id,endpoint_name,unit,quarantined_at,file_name,sha256
2,EP4YIFHV,host-a616,unit-lima,2026-09-10 13:30:00,load-ec-deps.bundle.js,f2bf148b4d999cb41ec764960ee718e0c9986a749f3b7b...
11,EPGSQVEB,host-9a4f,unit-bravo,2026-09-10 15:38:00,Final_Documents.zip,404ee57531c33156d62154f49e3700c8c1f09cd0c75692...
28,EPT3KTUL,host-ad7a,unit-quebec,2026-09-10 17:06:00,Dash Camera-49600.00.pdf,7505de4759aadebf5e21b2cc0f7f13fccdbec968e3b82b...
25,EPRD7FJT,host-916a,unit-romeo,2026-09-11 19:22:00,12th Plan Works Dashboard.pdf,5c7fc859c5a8f2bba02446c30020bb2b478686d66b5668...
18,EPMVZYZO,host-ba01,unit-quebec,2026-09-12 09:23:00,mydoc-75.pdf,92e24303edb5f2b6f1bc6c0c375655717eb2227c1622c4...
7,EPDRB236,host-b3df,unit-hotel,2026-09-12 13:18:00,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...
13,EPJ76RZV,host-aedd,unit-alpha,2026-09-14 08:28:00,COMPRESSED FINAL COMPARISION RETIREES WELFARE1...,d4a23078663777e758c04d2670b0d6aac8aa7ba5f91f81...
29,EPVC2P55,host-aca0,unit-alpha,2026-09-14 13:25:00,Data1.cab,925b63ce20d94bc4a0d98d02d153161dbba9ac61c39ec3...
15,EPLDM6PE,host-2d83,unit-foxtrot,2026-09-16 13:30:00,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...
5,EPAJI3E5,host-f6a4,unit-oscar,2026-09-17 12:44:00,Lakeside traders.docx,c11e86d9e30b5fad8daf1c15c4dbbfd8d31b581a5d930b...


In [12]:
# ============================================================
# WEB ACTIVITY ANALYSIS
# Find activity for the 13 endpoints first quarantined
# during the 10-21 September investigation window
# ============================================================

web_files = sorted(DATA_DIR.glob("web_activity_part*.csv"))

target_endpoints = set(window_quarantines_clean["endpoint_id"])

print("Target endpoints:", len(target_endpoints))
print(sorted(target_endpoints))

web_matches = []

for web_file in web_files:
    print(f"\nProcessing: {web_file.name}")

    for chunk in pd.read_csv(
        web_file,
        chunksize=200_000,
        low_memory=False
    ):
        matches = chunk[chunk["endpoint_id"].isin(target_endpoints)]

        if not matches.empty:
            web_matches.append(matches)

web_activity_target = pd.concat(
    web_matches,
    ignore_index=True
)

print("\nTotal matching web records:",
      len(web_activity_target))

print("Endpoints found:",
      web_activity_target["endpoint_id"].nunique())

display(web_activity_target.head(20))

Target endpoints: 13
['EP3GALVS', 'EP4YIFHV', 'EPAJI3E5', 'EPDRB236', 'EPGSQVEB', 'EPJ76RZV', 'EPLDM6PE', 'EPMVZYZO', 'EPNPX6VY', 'EPRD7FJT', 'EPSES7ZK', 'EPT3KTUL', 'EPVC2P55']

Processing: web_activity_part1.csv

Processing: web_activity_part2.csv

Processing: web_activity_part3.csv

Processing: web_activity_part4.csv

Processing: web_activity_part5.csv

Total matching web records: 142333
Endpoints found: 13


,endpoint_id,endpoint_name,mac,unit,log_type,event_time,received_time,message
0,EPLDM6PE,host-2d83,26:F7:A0:14:54:99,unit-foxtrot,web,10-09-2026 00:30:05,10-09-2026 11:12:19,198.19.153.92
1,EPLDM6PE,host-2d83,26:F7:A0:14:54:99,unit-foxtrot,web,10-09-2026 00:30:05,10-09-2026 11:12:19,n22ce.svc-72dec3.example
2,EPLDM6PE,host-2d83,26:F7:A0:14:54:99,unit-foxtrot,web,10-09-2026 00:30:05,10-09-2026 11:12:19,google.com
3,EPLDM6PE,host-2d83,26:F7:A0:14:54:99,unit-foxtrot,web,10-09-2026 00:30:05,10-09-2026 11:12:19,dns.google
4,EPLDM6PE,host-2d83,26:F7:A0:14:54:99,unit-foxtrot,web,10-09-2026 00:30:05,10-09-2026 11:12:19,8.8.4.4
5,EPLDM6PE,host-2d83,26:F7:A0:14:54:99,unit-foxtrot,web,10-09-2026 00:30:05,10-09-2026 11:12:19,8.8.8.8
6,EPLDM6PE,host-2d83,26:F7:A0:14:54:99,unit-foxtrot,web,10-09-2026 00:30:05,10-09-2026 11:12:19,192.178.211.102
7,EPLDM6PE,host-2d83,26:F7:A0:14:54:99,unit-foxtrot,web,10-09-2026 00:30:05,10-09-2026 11:12:19,192.178.211.101
8,EPNPX6VY,host-6ef6,82:FB:DF:E6:F2:88,unit-juliett,web,10-09-2026 00:30:08,10-09-2026 20:52:47,142.250.205.78
9,EPNPX6VY,host-6ef6,82:FB:DF:E6:F2:88,unit-juliett,web,10-09-2026 00:30:08,10-09-2026 20:52:47,google.com


In [13]:
# ============================================================
# WEB-BLOCK CORRELATION
# Find destinations blocked by the web filter for our
# 13 endpoints under investigation.
# ============================================================

web_blocks = security_df[
    (security_df["log_type"] == "web_block") &
    (security_df["endpoint_id"].isin(target_endpoints))
].copy()

print("Web-block events for target endpoints:", len(web_blocks))
print("Endpoints with web blocks:", web_blocks["endpoint_id"].nunique())

# Clean table
web_blocks_clean = web_blocks[
    [
        "endpoint_id",
        "endpoint_name",
        "unit",
        "event_time",
        "message"
    ]
].sort_values("event_time")

display(web_blocks_clean)

Web-block events for target endpoints: 123
Endpoints with web blocks: 7


,endpoint_id,endpoint_name,unit,event_time,message
5753,EPT3KTUL,host-ad7a,unit-quebec,06-09-2026 10:21:10,sync.aralego.com
5436,EPAJI3E5,host-f6a4,unit-oscar,07-09-2026 20:00:20,sync.aralego.com
12567,EPGSQVEB,host-9a4f,unit-bravo,09-09-2026 12:38:12,www.googleapis.com
13675,EPGSQVEB,host-9a4f,unit-bravo,09-09-2026 12:38:12,www.googleapis.com
10224,EPGSQVEB,host-9a4f,unit-bravo,09-09-2026 12:38:12,www.googleapis.com
...,...,...,...,...,...
22607,EPGSQVEB,host-9a4f,unit-bravo,26-09-2026 10:03:10,www.googleapis.com
22609,EPGSQVEB,host-9a4f,unit-bravo,26-09-2026 10:14:10,www.googleapis.com
22610,EPGSQVEB,host-9a4f,unit-bravo,26-09-2026 10:37:10,www.googleapis.com
22816,EPGSQVEB,host-9a4f,unit-bravo,26-09-2026 12:14:12,www.googleapis.com


In [14]:
# ============================================================
# FORENSIC SUMMARY OF THE 13 TARGET ENDPOINTS
# ============================================================

# Count web activity per endpoint
web_counts = (
    web_activity_target
    .groupby("endpoint_id")
    .size()
    .rename("web_events")
)

# Count blocked events per endpoint
block_counts = (
    web_blocks
    .groupby("endpoint_id")
    .size()
    .rename("web_blocks")
)

# Start with quarantine information
summary = window_quarantines_clean[
    [
        "endpoint_id",
        "endpoint_name",
        "unit",
        "quarantined_at",
        "file_name",
        "sha256"
    ]
].copy()

# Add web activity counts
summary = summary.merge(
    web_counts,
    on="endpoint_id",
    how="left"
)

summary = summary.merge(
    block_counts,
    on="endpoint_id",
    how="left"
)

summary["web_events"] = summary["web_events"].fillna(0).astype(int)
summary["web_blocks"] = summary["web_blocks"].fillna(0).astype(int)

summary = summary.sort_values("quarantined_at")

display(summary)

,endpoint_id,endpoint_name,unit,quarantined_at,file_name,sha256,web_events,web_blocks
0,EP4YIFHV,host-a616,unit-lima,2026-09-10 13:30:00,load-ec-deps.bundle.js,f2bf148b4d999cb41ec764960ee718e0c9986a749f3b7b...,1034,0
1,EPGSQVEB,host-9a4f,unit-bravo,2026-09-10 15:38:00,Final_Documents.zip,404ee57531c33156d62154f49e3700c8c1f09cd0c75692...,9256,53
2,EPT3KTUL,host-ad7a,unit-quebec,2026-09-10 17:06:00,Dash Camera-49600.00.pdf,7505de4759aadebf5e21b2cc0f7f13fccdbec968e3b82b...,6358,1
3,EPRD7FJT,host-916a,unit-romeo,2026-09-11 19:22:00,12th Plan Works Dashboard.pdf,5c7fc859c5a8f2bba02446c30020bb2b478686d66b5668...,12656,0
4,EPMVZYZO,host-ba01,unit-quebec,2026-09-12 09:23:00,mydoc-75.pdf,92e24303edb5f2b6f1bc6c0c375655717eb2227c1622c4...,7557,1
5,EPDRB236,host-b3df,unit-hotel,2026-09-12 13:18:00,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,912,0
6,EPJ76RZV,host-aedd,unit-alpha,2026-09-14 08:28:00,COMPRESSED FINAL COMPARISION RETIREES WELFARE1...,d4a23078663777e758c04d2670b0d6aac8aa7ba5f91f81...,4188,0
7,EPVC2P55,host-aca0,unit-alpha,2026-09-14 13:25:00,Data1.cab,925b63ce20d94bc4a0d98d02d153161dbba9ac61c39ec3...,6209,3
8,EPLDM6PE,host-2d83,unit-foxtrot,2026-09-16 13:30:00,recently-used.xbel,5e10b3490160cca0fc872f2b026d4b40bafc236852f83d...,16364,0
9,EPAJI3E5,host-f6a4,unit-oscar,2026-09-17 12:44:00,Lakeside traders.docx,c11e86d9e30b5fad8daf1c15c4dbbfd8d31b581a5d930b...,27333,1


In [15]:
# ============================================================
# CORRELATE WEB BLOCKS WITH MALICIOUS QUARANTINE
# Look for blocked destinations in the 7 days before
# each endpoint's first malicious quarantine.
# ============================================================

# Convert web-block event time
web_blocks["event_time_dt"] = pd.to_datetime(
    web_blocks["event_time"],
    format="%d-%m-%Y %H:%M:%S",
    errors="coerce"
)

# First quarantine time for each target endpoint
target_quarantine_times = window_quarantines_clean[
    ["endpoint_id", "quarantined_at"]
].copy()

# Join quarantine time onto every block
block_correlation = web_blocks.merge(
    target_quarantine_times,
    on="endpoint_id",
    how="inner"
)

# Calculate difference between block and quarantine
block_correlation["hours_before_quarantine"] = (
    block_correlation["quarantined_at"]
    - block_correlation["event_time_dt"]
).dt.total_seconds() / 3600

# Keep blocks occurring 0–7 days before quarantine
relevant_blocks = block_correlation[
    (block_correlation["hours_before_quarantine"] >= 0) &
    (block_correlation["hours_before_quarantine"] <= 7 * 24)
].copy()

# Clean evidence table
relevant_blocks = relevant_blocks[
    [
        "endpoint_id",
        "endpoint_name",
        "unit",
        "event_time",
        "message",
        "quarantined_at",
        "hours_before_quarantine"
    ]
].sort_values(
    ["endpoint_id", "quarantined_at", "event_time"]
)

print("Web blocks within 7 days before first malicious quarantine:",
      len(relevant_blocks))

print("Endpoints with relevant blocks:",
      relevant_blocks["endpoint_id"].nunique())

display(relevant_blocks)

Web blocks within 7 days before first malicious quarantine: 26
Endpoints with relevant blocks: 5


,endpoint_id,endpoint_name,unit,event_time,message,quarantined_at,hours_before_quarantine
7,EPGSQVEB,host-9a4f,unit-bravo,09-09-2026 12:38:12,www.googleapis.com,2026-09-10 15:38:00,26.996667
12,EPGSQVEB,host-9a4f,unit-bravo,09-09-2026 12:38:12,www.googleapis.com,2026-09-10 15:38:00,26.996667
30,EPGSQVEB,host-9a4f,unit-bravo,09-09-2026 12:38:12,www.googleapis.com,2026-09-10 15:38:00,26.996667
39,EPGSQVEB,host-9a4f,unit-bravo,09-09-2026 12:38:12,www.googleapis.com,2026-09-10 15:38:00,26.996667
42,EPGSQVEB,host-9a4f,unit-bravo,09-09-2026 12:38:12,www.googleapis.com,2026-09-10 15:38:00,26.996667
45,EPGSQVEB,host-9a4f,unit-bravo,09-09-2026 12:38:12,www.googleapis.com,2026-09-10 15:38:00,26.996667
53,EPGSQVEB,host-9a4f,unit-bravo,09-09-2026 12:38:12,www.googleapis.com,2026-09-10 15:38:00,26.996667
8,EPGSQVEB,host-9a4f,unit-bravo,09-09-2026 12:39:12,www.googleapis.com,2026-09-10 15:38:00,26.980000
11,EPGSQVEB,host-9a4f,unit-bravo,09-09-2026 12:39:12,www.googleapis.com,2026-09-10 15:38:00,26.980000
31,EPGSQVEB,host-9a4f,unit-bravo,09-09-2026 12:39:12,www.googleapis.com,2026-09-10 15:38:00,26.980000


In [16]:
# ============================================================
# AV DETECTION / QUARANTINE EVIDENCE
# Extract actual AV actions for the 13 target endpoints.
# ============================================================

target_av = security_df[
    (security_df["log_type"] == "av_scan") &
    (security_df["endpoint_id"].isin(target_endpoints)) &
    (
        security_df["message"].str.contains(
            "Infected files found|Processing:|Attempting to handle:|Quarantined and deleted:",
            case=False,
            na=False
        )
    )
].copy()

# Extract file paths from AV messages
target_av["file_path"] = target_av["message"].str.extract(
    r"(?:Processing:|Attempting to handle:|Quarantined and deleted:)\s*'?([^']+)",
    expand=False
)

# Keep useful evidence columns
av_evidence = target_av[
    [
        "endpoint_id",
        "endpoint_name",
        "unit",
        "event_time",
        "message",
        "file_path"
    ]
].sort_values(
    ["endpoint_id", "event_time"]
)

print("AV action records for target endpoints:", len(av_evidence))
print("Endpoints with AV action records:",
      av_evidence["endpoint_id"].nunique())

display(av_evidence)

AV action records for target endpoints: 31
Endpoints with AV action records: 4


,endpoint_id,endpoint_name,unit,event_time,message,file_path
17964,EP3GALVS,host-58bb,unit-golf,19-09-2026 01:15:01,Infected file is Infected files found. Beginni...,NaN
17965,EP3GALVS,host-58bb,unit-golf,19-09-2026 01:15:01,Infected file is Attempting to handle: /home/u...,/home/usera46a/.local/share/recently-used.xbel
17966,EP3GALVS,host-58bb,unit-golf,19-09-2026 01:15:01,Infected file is Processing: '/home/usera46a/....,/home/usera46a/.local/share/recently-used.xbel
17967,EP3GALVS,host-58bb,unit-golf,19-09-2026 01:15:01,Infected file is Quarantined and deleted: /hom...,/home/usera46a/.local/share/recently-used.xbel...
10337,EPDRB236,host-b3df,unit-hotel,12-09-2026 01:15:01,Infected file is Processing: '/home/user/.loca...,/home/user/.local/share/recently-used.xbel
10338,EPDRB236,host-b3df,unit-hotel,12-09-2026 01:15:01,Infected file is Quarantined and deleted: /hom...,/home/user/.local/share/recently-used.xbel to ...
10339,EPDRB236,host-b3df,unit-hotel,12-09-2026 01:15:01,Infected file is Attempting to handle: /home/u...,/home/user/.local/share/recently-used.xbel
10340,EPDRB236,host-b3df,unit-hotel,12-09-2026 01:15:01,Infected file is Infected files found. Beginni...,NaN
9254,EPGSQVEB,host-9a4f,unit-bravo,10-09-2026 01:15:01,Infected file is Infected files found. Beginni...,NaN
9255,EPGSQVEB,host-9a4f,unit-bravo,10-09-2026 01:15:01,Infected file is Attempting to handle: /home/u...,/home/user6405/Desktop/DOWNLOAD FOLDER/Project...


In [17]:
# ============================================================
# STEP 11B — CLEAN AV QUARANTINE EVIDENCE
# One row per endpoint + quarantined file
# ============================================================

av_quarantine = target_av[
    target_av["message"].str.contains(
        "Quarantined and deleted:",
        case=False,
        na=False
    )
].copy()

# Extract the original file path
av_quarantine["quarantined_file"] = av_quarantine["message"].str.extract(
    r"Quarantined and deleted:\s*(.*?)\s+to\s+",
    expand=False
)

# Clean output
av_quarantine_clean = (
    av_quarantine[
        [
            "endpoint_id",
            "endpoint_name",
            "unit",
            "event_time",
            "quarantined_file"
        ]
    ]
    .drop_duplicates()
    .sort_values(["event_time", "endpoint_id"])
    .reset_index(drop=True)
)

print("Confirmed AV quarantine records:", len(av_quarantine_clean))
print("Endpoints with confirmed AV quarantine:",
      av_quarantine_clean["endpoint_id"].nunique())

display(av_quarantine_clean)

Confirmed AV quarantine records: 9
Endpoints with confirmed AV quarantine: 4


,endpoint_id,endpoint_name,unit,event_time,quarantined_file
0,EPGSQVEB,host-9a4f,unit-bravo,10-09-2026 01:15:01,/home/user6405/Desktop/DOWNLOAD FOLDER/Project...
1,EPGSQVEB,host-9a4f,unit-bravo,10-09-2026 01:15:01,/home/user6405/Desktop/DOWNLOAD FOLDER/Documen...
2,EPDRB236,host-b3df,unit-hotel,12-09-2026 01:15:01,/home/user/.local/share/recently-used.xbel
3,EP3GALVS,host-58bb,unit-golf,19-09-2026 01:15:01,/home/usera46a/.local/share/recently-used.xbel
4,EPSES7ZK,host-4c8d,unit-papa,21-09-2026 01:15:01,/home/userd0ab/Downloads/hp_LJ1020_Full_Soluti...
5,EPSES7ZK,host-4c8d,unit-papa,21-09-2026 01:15:01,/home/userd0ab/Downloads/hp_LJ1020_Full_Soluti...
6,EPSES7ZK,host-4c8d,unit-papa,21-09-2026 01:15:01,/home/userd0ab/Downloads/hp_LJ1020_Full_Soluti...
7,EPSES7ZK,host-4c8d,unit-papa,21-09-2026 01:15:01,/home/userd0ab/Downloads/hp_LJ1020_Full_Soluti...
8,EPSES7ZK,host-4c8d,unit-papa,21-09-2026 01:15:01,/home/userd0ab/Downloads/hp_LJ1020_Full_Soluti...


In [19]:

av_summary = (
    av_quarantine_clean
    .groupby(
        ["endpoint_id", "endpoint_name", "unit", "event_time"],
        as_index=False
    )
    .agg(
        quarantined_files=("quarantined_file", "count")
    )
    .sort_values("event_time")
)

print("Endpoints with confirmed AV quarantine:",
      av_summary["endpoint_id"].nunique())

display(av_summary)

Endpoints with confirmed AV quarantine: 4


,endpoint_id,endpoint_name,unit,event_time,quarantined_files
2,EPGSQVEB,host-9a4f,unit-bravo,10-09-2026 01:15:01,2
1,EPDRB236,host-b3df,unit-hotel,12-09-2026 01:15:01,1
0,EP3GALVS,host-58bb,unit-golf,19-09-2026 01:15:01,1
3,EPSES7ZK,host-4c8d,unit-papa,21-09-2026 01:15:01,5


In [20]:
# ============================================================
# STEP 12 — WEB ACTIVITY BEFORE CONFIRMED AV QUARANTINE
# ============================================================

av_endpoints = av_summary["endpoint_id"].unique()

# Use the AV event time as the reference point
av_reference = (
    av_summary[
        ["endpoint_id", "event_time"]
    ]
    .drop_duplicates()
    .copy()
)

av_reference["event_time"] = pd.to_datetime(
    av_reference["event_time"],
    dayfirst=True,
    errors="coerce"
)

web_activity["event_time"] = pd.to_datetime(
    web_activity["event_time"],
    dayfirst=True,
    errors="coerce"
)

# Merge AV reference time with web activity
web_before_av = web_activity.merge(
    av_reference,
    on="endpoint_id",
    suffixes=("_web", "_av")
)

# Calculate time difference
web_before_av["hours_before_av"] = (
    web_before_av["event_time_av"] -
    web_before_av["event_time_web"]
).dt.total_seconds() / 3600

# Keep activity within 24 hours BEFORE AV detection
web_before_av = web_before_av[
    (web_before_av["hours_before_av"] >= 0) &
    (web_before_av["hours_before_av"] <= 24)
].copy()

print("Web events within 24 hours before confirmed AV quarantine:",
      len(web_before_av))

print("Endpoints covered:",
      web_before_av["endpoint_id"].nunique())

display(
    web_before_av[
        [
            "endpoint_id",
            "event_time_web",
            "destination",
            "hours_before_av"
        ]
    ]
    .sort_values(["endpoint_id", "event_time_web"])
    .head(50)
)

NameError: name 'web_activity' is not defined

In [22]:
# ============================================================
# FIND AVAILABLE DATAFRAMES
# ============================================================

import pandas as pd

df_names = list(globals().keys())

for name in df_names:
    obj = globals()[name]
    if isinstance(obj, pd.DataFrame):
        print(f"{name}: {obj.shape}")

security_df: (23540, 9)
usb_df: (8, 8)
malicious_quarantine: (134, 10)
compromised_summary: (33, 5)
first_malicious: (33, 9)
window_compromised: (14, 9)
first_quarantine: (33, 7)
clean_quarantine: (33, 6)
window_quarantines_clean: (13, 6)
chunk: (162424, 8)
matches: (8145, 8)
web_activity_target: (142333, 8)
web_blocks: (123, 9)
web_blocks_clean: (123, 5)
summary: (13, 8)
target_quarantine_times: (13, 2)
block_correlation: (123, 11)
relevant_blocks: (26, 7)
target_av: (31, 10)
av_evidence: (31, 6)
av_quarantine: (9, 11)
av_quarantine_clean: (9, 5)
av_summary: (4, 5)
av_reference: (4, 2)


In [23]:
# ============================================================
# STEP 12 — WEB ACTIVITY BEFORE CONFIRMED AV QUARANTINE
# ============================================================

# Use the 4 endpoints with confirmed AV quarantine
av_endpoints = av_summary["endpoint_id"].unique()

# AV reference time
av_reference = (
    av_summary[
        ["endpoint_id", "event_time"]
    ]
    .drop_duplicates()
    .copy()
)

av_reference["event_time"] = pd.to_datetime(
    av_reference["event_time"],
    dayfirst=True,
    errors="coerce"
)

# Copy target web activity
web_before_av = web_activity_target.copy()

web_before_av["event_time"] = pd.to_datetime(
    web_before_av["event_time"],
    dayfirst=True,
    errors="coerce"
)

# Keep only the 4 AV-confirmed endpoints
web_before_av = web_before_av[
    web_before_av["endpoint_id"].isin(av_endpoints)
].copy()

# Match each endpoint with its AV reference time
web_before_av = web_before_av.merge(
    av_reference,
    on="endpoint_id",
    suffixes=("_web", "_av")
)

# Calculate hours before AV event
web_before_av["hours_before_av"] = (
    web_before_av["event_time_av"] -
    web_before_av["event_time_web"]
).dt.total_seconds() / 3600

# Keep only web activity in the 24 hours before AV detection
web_before_av = web_before_av[
    (web_before_av["hours_before_av"] >= 0) &
    (web_before_av["hours_before_av"] <= 24)
].copy()

print(
    "Web events within 24 hours before confirmed AV quarantine:",
    len(web_before_av)
)

print(
    "Endpoints covered:",
    web_before_av["endpoint_id"].nunique()
)

display(
    web_before_av[
        [
            "endpoint_id",
            "event_time_web",
            "destination",
            "hours_before_av"
        ]
    ]
    .sort_values(["endpoint_id", "event_time_web"])
    .head(50)
)

Web events within 24 hours before confirmed AV quarantine: 2838
Endpoints covered: 3


KeyError: "['destination'] not in index"

In [24]:
print(web_activity_target.columns.tolist())

['endpoint_id', 'endpoint_name', 'mac', 'unit', 'log_type', 'event_time', 'received_time', 'message']


In [25]:
# ============================================================
# STEP 12C — EXTRACT WEB DESTINATIONS
# ============================================================

# Extract the destination from the web activity message
web_before_av["destination"] = web_before_av["message"].str.extract(
    r"(?:https?://|www\.)([A-Za-z0-9.-]+)",
    expand=False
)

# If extraction fails, keep the original message
web_before_av["destination"] = web_before_av["destination"].fillna(
    web_before_av["message"].str[:100]
)

print(
    "Web events within 24 hours before confirmed AV quarantine:",
    len(web_before_av)
)

print(
    "Endpoints covered:",
    web_before_av["endpoint_id"].nunique()
)

display(
    web_before_av[
        [
            "endpoint_id",
            "event_time_web",
            "destination",
            "hours_before_av"
        ]
    ]
    .sort_values(["endpoint_id", "event_time_web"])
    .head(50)
)

Web events within 24 hours before confirmed AV quarantine: 2838
Endpoints covered: 3


,endpoint_id,event_time_web,destination,hours_before_av
27514,EP3GALVS,2026-09-18 08:36:57,198.18.4.194,16.634444
27515,EP3GALVS,2026-09-18 08:36:57,n9fa9.svc-347513.example,16.634444
27516,EP3GALVS,2026-09-18 08:36:57,n22ce.svc-72dec3.example,16.634444
27517,EP3GALVS,2026-09-18 08:36:57,198.19.217.201,16.634444
27518,EP3GALVS,2026-09-18 08:36:57,198.19.113.214,16.634444
27519,EP3GALVS,2026-09-18 08:36:57,198.19.153.92,16.634444
27520,EP3GALVS,2026-09-18 08:36:57,svc-1ca9c2.gov.example,16.634444
27521,EP3GALVS,2026-09-18 08:36:57,8.8.8.8,16.634444
27522,EP3GALVS,2026-09-18 08:36:57,svc-c06b6c.gov.example,16.634444
27523,EP3GALVS,2026-09-18 08:36:57,dns.google,16.634444


In [26]:
# ============================================================
# STEP 12D — LAST WEB EVENTS BEFORE AV DETECTION
# ============================================================

last_web_before_av = (
    web_before_av
    .sort_values(["endpoint_id", "event_time_web"])
    .groupby("endpoint_id")
    .tail(10)
    .copy()
)

# Keep the most useful columns
last_web_before_av = last_web_before_av[
    [
        "endpoint_id",
        "event_time_web",
        "destination",
        "hours_before_av"
    ]
].sort_values(
    ["endpoint_id", "event_time_web"]
)

display(last_web_before_av)

,endpoint_id,event_time_web,destination,hours_before_av
30084,EP3GALVS,2026-09-18 12:02:37,8.8.4.4,13.206667
30085,EP3GALVS,2026-09-18 12:02:37,dns.google,13.206667
30328,EP3GALVS,2026-09-18 12:34:12,8.8.8.8,12.680278
30329,EP3GALVS,2026-09-18 12:34:12,dns.google,12.680278
30330,EP3GALVS,2026-09-18 12:34:12,8.8.4.4,12.680278
30331,EP3GALVS,2026-09-18 12:34:12,n22ce.svc-72dec3.example,12.680278
30332,EP3GALVS,2026-09-18 12:34:12,svc-384d89.example,12.680278
30333,EP3GALVS,2026-09-18 12:34:12,n9fa9.svc-347513.example,12.680278
30334,EP3GALVS,2026-09-18 12:34:12,198.19.153.92,12.680278
30335,EP3GALVS,2026-09-18 12:34:12,198.19.113.214,12.680278


In [27]:
# ============================================================
# STEP 12E — FINAL WEB / AV CORRELATION
# One latest web event per AV-confirmed endpoint
# ============================================================

latest_web_before_av = (
    web_before_av
    .sort_values(["endpoint_id", "event_time_web"])
    .groupby("endpoint_id")
    .tail(1)
    .copy()
)

latest_web_before_av = latest_web_before_av[
    [
        "endpoint_id",
        "event_time_web",
        "destination",
        "hours_before_av"
    ]
].sort_values("endpoint_id")

display(latest_web_before_av)

,endpoint_id,event_time_web,destination,hours_before_av
30335,EP3GALVS,2026-09-18 12:34:12,198.19.113.214,12.680278
6883,EPDRB236,2026-09-11 12:12:12,google.com,13.046944
39464,EPSES7ZK,2026-09-20 12:41:35,192.29.69.178,12.557222


In [28]:
# ============================================================
# STEP 13A — FINAL INVESTIGATION FINDINGS
# ============================================================

findings = av_summary[
    [
        "endpoint_id",
        "endpoint_name",
        "unit",
        "event_time",
        "quarantined_files"
    ]
].copy()

# Add latest web activity where available
findings = findings.merge(
    latest_web_before_av[
        [
            "endpoint_id",
            "event_time_web",
            "destination",
            "hours_before_av"
        ]
    ],
    on="endpoint_id",
    how="left"
)

# Rename columns for report readability
findings = findings.rename(
    columns={
        "event_time": "av_event_time",
        "event_time_web": "latest_web_time",
        "hours_before_av": "hours_before_av"
    }
)

# Show clean table
display(
    findings.sort_values("av_event_time")
)

,endpoint_id,endpoint_name,unit,av_event_time,quarantined_files,latest_web_time,destination,hours_before_av
0,EPGSQVEB,host-9a4f,unit-bravo,10-09-2026 01:15:01,2,NaT,NaN,NaN
1,EPDRB236,host-b3df,unit-hotel,12-09-2026 01:15:01,1,2026-09-11 12:12:12,google.com,13.046944
2,EP3GALVS,host-58bb,unit-golf,19-09-2026 01:15:01,1,2026-09-18 12:34:12,198.19.113.214,12.680278
3,EPSES7ZK,host-4c8d,unit-papa,21-09-2026 01:15:01,5,2026-09-20 12:41:35,192.29.69.178,12.557222


In [29]:
# ============================================================
# STEP 13C — OVERALL INVESTIGATION STATISTICS
# ============================================================

total_endpoints = len(target_endpoints)

av_confirmed_endpoints = av_summary["endpoint_id"].nunique()

total_av_quarantine_records = len(av_quarantine_clean)

web_correlated_endpoints = latest_web_before_av["endpoint_id"].nunique()

web_block_endpoints = web_blocks_clean["endpoint_id"].nunique()

relevant_web_blocks = len(relevant_blocks)

print("==============================================")
print("        INVESTIGATION STATISTICS")
print("==============================================")
print(f"Target endpoints investigated       : {total_endpoints}")
print(f"Endpoints with AV quarantine        : {av_confirmed_endpoints}")
print(f"Confirmed AV quarantine records     : {total_av_quarantine_records}")
print(f"Endpoints with recent web activity  : {web_correlated_endpoints}")
print(f"Endpoints with web-block activity  : {web_block_endpoints}")
print(f"Relevant pre-quarantine web blocks  : {relevant_web_blocks}")
print("==============================================")

        INVESTIGATION STATISTICS
Target endpoints investigated       : 13
Endpoints with AV quarantine        : 4
Confirmed AV quarantine records     : 9
Endpoints with recent web activity  : 3
Endpoints with web-block activity  : 7
Relevant pre-quarantine web blocks  : 26
